## LANGCHAIN - STEP BY STEP

### 1. CONNECTING THE LLM

In [ ]:
# Install required packages. 
# Note: Using `%pip` and `-U` ensures proper upgrades in the active Colab kernel.
%pip install -U langchain "langchain[google-genai]"


## Agent = Model + Harness.
LangChain provides `create_agent`: a minimal and highly configurable harness. The harness is everything that surrounds the model's loop: the prompt, the tools, and any middleware that shapes the behavior. Start with primitives and compose exactly what your use case needs. It supports OpenAI, Anthropic, Google, and more.

------------------------------

## The "Agent Harness" Paradigm in LangChain
### 1. Clear and Technical Definition
In modern Artificial Intelligence application development, architecture has evolved from simple code "wrappers" to the concept of the Agent Harness.

Technically, a Harness is the infrastructure, execution environment, and control system that wraps around the LLM's inference cycle (the model loop). While the LLM functions as an isolated "cognitive engine," the harness acts as the "chassis and operating systems" that allow it to interact safely, structurally, and autonomously with the real world.

### 2. Engineering Layers in the Current Ecosystem
LLM development is now divided into three concentric levels of engineering:

   1. **Prompt Engineering**: Designing the exact text instructions the model receives.
   2. **Context Engineering**: Managing what specific data the model sees in its context window and when it sees it.
   3. **Harness Engineering (Current Focus)**: Encompasses the previous two and adds tool orchestration (Tool Calling), state persistence (memory), error management, verification loops, and system security.

### 3. Why is this the "New Way" to Connect LLMs?
Previously, connecting an LLM meant chaining components rigidly (the old sequential Chains in LangChain). The modern approach based on `create_agent` and harness abstractions offers:

- **Model Agnostic**: The harness abstracts the underlying API. You can swap the engine (OpenAI, Anthropic, Google) without rewriting your tool or memory logic.
- **Configurable Middleware**: Allows intercepting requests and responses to inject custom logic (e.g., cost control, history compression, or security) before and after the LLM acts.
- **Autonomous Loop Architecture**: Instead of executing a single request, the harness autonomously manages a repetitive cycle: evaluates the prompt → calls the LLM → executes tools if needed → processes results → queries the LLM again.


In [ ]:
import os

from langchain.agents import create_agent
from google.colab import userdata

# Fetch the API_KEY from environment variables (in this case, from a Google Colab notebook)
# This prevents exposing private keys when sharing the notebook (remember to set your own, e.g., Google's free tier)
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 1. Initialize the pure conversational harness
agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    system_prompt="You are an expert Artificial Intelligence tutor. Provide concise and clear explanations. Answer in English.",
)

# 2. The harness is executed by sending the conversation state (messages)
# The user interacts directly via structured text
result = agent.invoke(
    {
         "messages":
        [
            {
                 "role": "user",
                 "content": "Explain what an 'Agent Harness' is in one sentence."
            }
        ]
    }
)

# 3. Inspect the LLM Output
# Retrieve the last message from the history returned by the harness.
last_message = result["messages"][-1]

# Option A: Inspect the unified internal structure (content_blocks)
print("--- Content Blocks Structure ---")
# Full response: print(last_message.content_blocks)
# Clean response
print(f"AI 🤖: {last_message.content_blocks[0].get('text', '')}")

# Option B: Extract clean text (Recommended for production)
print("\n--- Final Text Output ---")
# Full response: print(last_message.content)
# Clean response
print(f"AI 🤖: {last_message.content[0].get('text', '')}")


In modern LangChain architecture, the `create_agent` method is designed with a minimalist philosophy: the harness should not duplicate the model's parameters.

Therefore, `create_agent` is divided into two types of properties: the structural parameters of the harness, and the generation hyperparameters (which are injected via configuration strings or dictionaries).

------------------------------
## 1. Structural Parameters (Directly in `create_agent`)
These are the native properties that define what the agent's harness can do:

- **`model`** (str | BaseChatModel): The unique identifier (e.g., `"google_genai:gemini-3.6-flash"`, `"openai:gpt-4o"`).
- **`system_prompt`** (str | SystemMessage): Defines system rules, role, constraints, and response language.
- **`tools`** (list[Callable | BaseTool]): List of native Python functions the agent can invoke in its operational loop.
- **`response_format`** (BaseModel | TypedDict | dict): (Crucial in 2026!) Forces the agent to return a structured format (using a Pydantic class) instead of plain text, ideal for automatic structured JSON extraction.
- **`middleware`** (list): List of layers or interceptors to audit costs, inject security, format history, or halt inappropriate executions.

------------------------------
## 2. Model Parameters (temperature, max_tokens, etc.)
To modify the cognitive parameters of the LLM (hyperparameters), they are not passed directly to `create_agent`, but configured in two ways depending on the course level:

### Method A: When initializing the model (For advanced local scripts)
If you pass the object instance instead of a string, you can configure the exact behavior. We will show an example of this method using LangChain's integration classes, in this case with `google-genai`.
https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai

### Method B: Using Models (`init_chat_model`)
In production, instantiating models directly (like `ChatGoogleGenerativeAI`) generates rigid code coupled to a single provider.

The modern and recommended syntax is to use `init_chat_model` to initialize the cognitive engine (for its configuration flexibility) and pass it immediately to the `create_agent` harness. This also allows you to change the AI provider via environment variables without modifying a single line of Python code (it is Model Agnostic).
https://docs.langchain.com/oss/python/langchain/models

### Method C: Using Harness Profiles
In current versions of LangChain, you can pass configuration dictionaries in the `.invoke()` method or register global profiles (`HarnessProfile`) so that the same `"google_genai:gemini..."` string applies dynamic restrictions on the fly without rebuilding the object. This method has a medium-advanced complexity level, so it will be covered in later chapters. If you wish to read more:
https://learn.microsoft.com/en-us/agent-framework/concepts/agents/running-agents?pivots=programming-language-python

------------------------------

### Comparison Table
You can add this summarized table for a quick reference of what each parameter controls:

| Parameter | Where is it configured? | Type | What does it control in the Agent? |
|---|---|---|---|
| `system_prompt` | `create_agent` | String | Personality, role, and base language. |
| `response_format` | `create_agent` | Pydantic Class | Forces structured responses (JSON with schema). |
| `temperature` | Model Instance | Float (0 to 1) | Randomness: 0.1 for code/data, 0.8 for fluid chat. |
| `max_tokens` | Model Instance | Integer | Maximum allowed length for the model's response. |
| `max_retries` | Model Instance | Integer | How many times to retry the call if a network error occurs. |

### COGNITIVE PARAMETERS (HYPERPARAMETERS)

Method A: When initializing the model (For advanced local scripts)

In [ ]:
%pip install -U langchain-google-genai

The recommendation is that if you decide to use the format with custom cognitive parameterization, use this template. It is identical to the one above, but uses the `langchain-google-genai` library, which you must install additionally to use the `ChatGoogleGenerativeAI` class in combination with agent harnesses. 

*Observation*: In the case of `gemini-3.6-flash`, it uses some default parameters. This will also happen with some versions of ChatGPT, Claude, etc. Always verify the specific characteristics and supported parameters of your provider's model.

In [ ]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from google.colab import userdata

# Fetch the API_KEY from environment variables (in this case, from a Google Colab notebook)
# This prevents exposing private keys when sharing the notebook
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Here we configure the individual hyperparameters of the cognitive engine
custom_model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    # temperature=0.1,        # More deterministic (0.0) or more creative (1.0) - gemini 3.6 uses default parameters
    max_tokens=500,           # Strict limit on response size
    # top_p=0.95,             # Nucleus sampling - gemini 3.6 uses default parameters
    max_retries=3             # Automatic retries if the Google API fails
)

agent = create_agent(
    model=custom_model,
    system_prompt="You are an expert Artificial Intelligence tutor. Provide concise and clear explanations. Answer in English.",
)

result = agent.invoke(
    {
         "messages":
        [
            {
                 "role": "user",
                 "content": "Explain what an 'Agent Harness' is in one sentence."
            }
        ]
    }
)

# Inspect the LLM Output
# Retrieve the last message from the history returned by the harness.
last_message = result["messages"][-1]

# Option A: Inspect the unified internal structure (content_blocks)
print("--- Content Blocks Structure ---")
print(f"AI 🤖: {last_message.content_blocks[0].get('text', '')}")

# Option B: Extract clean text (Recommended for production)
print("\n--- Final Text Output ---")
print(f"AI 🤖: {last_message.content[0].get('text', '')}")

Method B: In combination with Models (`init_chat_model`)

LangChain's standard model interfaces allow you to access integrations with various providers, making it easy to experiment with different models and switch between them to find the best fit for your use case.

**Basic Usage**

Models can be used in two ways:
- **With Agents**: Models can be specified dynamically when creating an agent.
- **Independently**: Models can be invoked directly (outside the agent loop) for tasks like text generation, classification, or extraction, without needing an agent framework.

The same model interface works in both contexts, giving you the flexibility to start with something simple and scale up to more complex agent-based workflows as needed.

**Initialize a model**
The easiest way to get started with a standalone model in LangChain is to use `init_chat_model` to initialize one from your preferred chat model provider.

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

# 1. API_KEY from a Google Colab notebook
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 2. THE MOTOR (Configured globally via production-ready initializer)
# If you want to switch to OpenAI tomorrow, just change the string to "openai:gpt-4o"
# and you won't have to change any imports at the top of the file.
production_model = init_chat_model(
    model=os.getenv("GEMINI_MODEL_NAME", "google_genai:gemini-3.6-flash"),
    temperature=0.3,
    max_retries=5,       # Fault-tolerant network
    timeout=60,          # Latency control in production
)

# 3. THE HARNESS (The runtime infrastructure surrounding the motor)
agent = create_agent(
    model=production_model, # We fuse the flexible engine into the harness
    system_prompt="You are an expert AI tutor. Provide concise explanations. Respond in English.",
)

# 4. Clean execution
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain what an 'Agent Harness' is."}]}
)

print(f"AI 🤖: {response['messages'][-1].content}")

------------------------------
## The Technical Difference: Engine vs. Chassis
### 1. The first approach (`init_chat_model`): The Pure Engine
`init_chat_model` (or instantiating `ChatGoogleGenerativeAI`) creates only the Language Model (LLM) object.
- **What it does**: It simply sends text to the API (OpenAI/Google) and waits for the text response. It is a flat round-trip channel.
- **Limitations**: It has no automatic memory, does not know how to react if the model decides it needs to use a tool (Tool Calling), and has no loop to repeat questions if the AI makes a mistake.

### 2. The second approach (`create_agent`): The Harness or Chassis
`create_agent` creates the complete Agent (The execution loop or Harness).
- **What it does**: Wraps the engine (LLM) and adds superpowers: tool orchestration, automatic `system_prompt` injection, context schemas (`context_schema`), and persistence with checkpointer.
- **Advantage**: If the model responds saying "I want to use the weather tool", the harness stops the model, executes the Python function underneath, takes the result, re-injects it into the model, and continues the flow without you writing extra code.

------------------------------
## The Best Alternative for Production: Combining Both
In production, instantiating models directly (like `ChatGoogleGenerativeAI`) generates rigid code coupled to a single provider. The modern and recommended syntax is to use `init_chat_model` to initialize the cognitive engine (for its configuration flexibility) and pass it immediately to the `create_agent` harness.

### Why is `init_chat_model` superior in production?
Because it allows you to change the AI provider using environment variables without modifying a single line of Python code (it is Model Agnostic).

## Comparison Table

| Feature | `init_chat_model` (LLM Only) | `create_agent` (Harness) | The Combination (Production) |
|---|---|---|---|
| Handles Tools automatically? | No ❌ (You must program the loop manually) | Yes ✅ | Yes ✅ |
| Supports checkpointer / Memory? | No ❌ | Yes ✅ | Yes ✅ |
| Change provider without changing code? | Yes ✅ (Via strings) | No, if using the direct class | Yes, Maximum flexibility ✅ |
| Main Purpose | Clean and agnostic initialization of the LLM. | Management of flow, rules, and middleware. | The ultimate engineering standard. |